# 01 获取数据与构造样本

研究范围是2020—2025年逐日沪深300成员；2019年第四季度只提供预热行情。

## 1. 设置日期与字段

In [ ]:
from jqdata import *
from pathlib import Path
import gc
import numpy as np
import pandas as pd

OUTPUT = Path('data/ml_alpha_simple')
OUTPUT.mkdir(parents=True, exist_ok=True)
INDEX = '000300.XSHG'
PRICE_START = '2019-10-01'
SIGNAL_START = '2020-01-01'
END = '2025-12-31'
BATCH_SIZE = 50
FIELDS = ['open', 'high', 'low', 'close', 'volume', 'money', 'paused']
FEATURES = ['momentum_2', 'momentum_5', 'volatility_5', 'volume_ratio_5', 'intraday_return']


## 2. 交易日历

price_days包含预热期；signal_days只包含研究区间。后面的shift按交易日行位置移动。

In [ ]:
price_days = pd.DatetimeIndex(get_trade_days(start_date=PRICE_START, end_date=END))
signal_days = price_days[price_days >= pd.Timestamp(SIGNAL_START)]
assert price_days.is_monotonic_increasing and not price_days.duplicated().any()
assert len(signal_days) > 0
print('行情交易日：', len(price_days), '研究交易日：', len(signal_days))


## 3. 每一天的指数成员

universe每行表示某只股票在某日属于指数。股票并集只决定下载哪些股票，不能替代当天成员资格。
这一节逐日查询，可能需要较长时间。

In [ ]:
member_parts = []
for i, day in enumerate(signal_days, start=1):
    stocks = get_index_stocks(INDEX, date=day.strftime('%Y-%m-%d'))
    assert len(stocks) == len(set(stocks)) == 300, '成员数量异常'
    member_parts.append(pd.DataFrame({'date': pd.Timestamp(day), 'stock': stocks}))


universe = pd.concat(member_parts, ignore_index=True)
assert not universe.duplicated(['date', 'stock']).any()
assert pd.DatetimeIndex(universe['date'].unique()).equals(signal_days)
assert universe.groupby('date')['stock'].nunique().eq(300).all()
download_stocks = sorted(universe['stock'].unique().tolist())
del member_parts
print('成员记录：', len(universe), '需要下载的股票：', len(download_stocks))
display(universe.head())


## 4. 原始行情和后复权因子

按年份、每批50只股票查询。原始行情不复权，factor使用后复权口径。
每批连接后检查日期×股票网格，只在内存中汇总，不保存中间批次。

In [ ]:
periods = [('2019-10-01', '2020-12-31')] + [
    ('{}-01-01'.format(year), '{}-12-31'.format(year)) for year in range(2021, 2026)
]
market_parts = []
for start, end in periods:
    days = price_days[(price_days >= pd.Timestamp(start)) & (price_days <= pd.Timestamp(end))]
    for offset in range(0, len(download_stocks), BATCH_SIZE):
        stocks = download_stocks[offset:offset + BATCH_SIZE]
        raw = get_price(stocks, start_date=start, end_date=end, frequency='daily',
                        fields=FIELDS, skip_paused=False, fq=None, panel=False, round=False)
        factors = get_price(stocks, start_date=start, end_date=end, frequency='daily',
                            fields=['factor'], skip_paused=False, fq='post', panel=False, round=False)
        raw = raw.rename(columns={'time': 'date', 'code': 'stock'})
        factors = factors.rename(columns={'time': 'date', 'code': 'stock', 'factor': 'post_factor'})
        raw['date'] = pd.to_datetime(raw['date'])
        factors['date'] = pd.to_datetime(factors['date'])
        part = raw.merge(factors[['date', 'stock', 'post_factor']],
                         on=['date', 'stock'], how='outer', validate='one_to_one', indicator=True)
        assert part['_merge'].eq('both').all(), '行情与因子没有完全对应'
        part = part.drop(columns=['_merge'])
        expected = pd.MultiIndex.from_product([days, stocks], names=['date', 'stock'])
        actual = pd.MultiIndex.from_arrays([part['date'], part['stock']], names=['date', 'stock'])
        assert len(expected.difference(actual)) == len(actual.difference(expected)) == 0, '行情网格异常'
        market_parts.append(part)
        print(start, end, '本批股票：', len(stocks), '行数：', len(part), flush=True)

market = pd.concat(market_parts, ignore_index=True).sort_values(['stock', 'date']).reset_index(drop=True)
assert not market.duplicated(['date', 'stock']).any()
del market_parts, raw, factors, part, expected, actual
gc.collect()
print('行情表形状：', market.shape)
display(market.head())


## 5. 清洗前核查

不凭空补价格，不删除停牌行或上市前、退市后的占位行。
保留完整交易日网格，才能让shift代表固定交易日，而不是跳过缺失后的下一条有效记录。

In [ ]:
has_price = market['close'].notna()
prices = market[['open', 'high', 'low', 'close']]
assert np.isfinite(prices.loc[has_price].values).all(), '有效行情的OHLC不完整'
assert not np.isinf(prices.values).any()
assert not (prices.le(0) & prices.notna()).any().any(), '存在非正价格'
bad_ohlc = (market['open'].lt(market['low']) | market['open'].gt(market['high'])
            | market['close'].lt(market['low']) | market['close'].gt(market['high']))
assert not bad_ohlc.any(), 'OHLC关系异常'
assert np.isfinite(market.loc[has_price, 'post_factor']).all()
assert market.loc[has_price, 'post_factor'].gt(0).all()
assert np.isfinite(market.loc[has_price, ['volume', 'money']].values).all()
assert not market[['volume', 'money']].lt(0).any().any()
assert market.loc[has_price, 'paused'].isin([0, 1]).all()
print('收盘价缺失：', market['close'].isna().sum())


In [ ]:
missing = market.loc[market['close'].isna(), ['date', 'stock']].copy()
security_records = []
for stock in sorted(missing['stock'].unique()):
    info = get_security_info(stock)
    assert info is not None, '证券资料缺失：' + stock
    security_records.append({'stock': stock, 'listing_date': pd.Timestamp(info.start_date),
                             'delisting_date': pd.Timestamp(info.end_date)})
security = pd.DataFrame(security_records, columns=['stock', 'listing_date', 'delisting_date'])
missing = missing.merge(security, on='stock', how='left', validate='many_to_one')
missing['reason'] = '待核查'
missing.loc[missing['date'] < missing['listing_date'], 'reason'] = '上市前'
missing.loc[missing['date'] > missing['delisting_date'], 'reason'] = '退市后'
print(missing['reason'].value_counts())
display(missing.head())
assert missing['reason'].isin(['上市前', '退市后']).all(), '有存续期内缺失，请先检查missing表'


## 6. 当天选股资格与复权价格

in_universe来自逐日成员表。后复权价格用于跨日期比较，不是实际成交价格。

In [ ]:
universe['in_universe'] = True
panel = market.merge(universe, on=['date', 'stock'], how='left', validate='one_to_one')
panel['in_universe'] = panel['in_universe'].fillna(False).astype(bool)
assert panel.loc[panel['in_universe'], ['close', 'post_factor']].notna().all().all()
panel = panel.sort_values(['stock', 'date']).reset_index(drop=True)
panel['adj_open'] = panel['open'] * panel['post_factor']
panel['adj_close'] = panel['close'] * panel['post_factor']
del market, universe, prices, missing, security, security_records, has_price, bad_ohlc
gc.collect()
print(panel['in_universe'].value_counts())


## 7. 五个特征

信号在收盘后生成，因此当天收盘价和成交量可用。所有shift与rolling都在同一股票内进行。

In [ ]:
panel['momentum_2'] = panel['adj_close'] / panel.groupby('stock')['adj_close'].shift(2) - 1
panel['momentum_5'] = panel['adj_close'] / panel.groupby('stock')['adj_close'].shift(5) - 1
panel['return_1'] = panel['adj_close'] / panel.groupby('stock')['adj_close'].shift(1) - 1
panel['volatility_5'] = panel.groupby('stock')['return_1'].transform(
    lambda s: s.rolling(5, min_periods=5).std())
past_volume_mean = panel.groupby('stock')['volume'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=5).mean())
panel['volume_ratio_5'] = panel['volume'] / past_volume_mean.where(past_volume_mean > 0)
panel['intraday_return'] = panel['adj_close'] / panel['adj_open'] - 1
print('成员日特征缺失：')
print(panel.loc[panel['in_universe'], FEATURES].isna().sum())
assert not np.isinf(panel[FEATURES].values).any(), '特征出现无穷值'
display(panel[['date', 'stock'] + FEATURES].head(8))


## 8. 标签与信号资格

label是下一交易日开盘到再下一交易日开盘的复权收益。
信号资格只依赖当时信息；不因为未来停牌或未来标签缺失而提前排除信号。

In [ ]:
panel['entry_adj_open'] = panel.groupby('stock')['adj_open'].shift(-1)
panel['exit_adj_open'] = panel.groupby('stock')['adj_open'].shift(-2)
panel['entry_date'] = panel.groupby('stock')['date'].shift(-1)
panel['label_end_date'] = panel.groupby('stock')['date'].shift(-2)
panel['label'] = panel['exit_adj_open'] / panel['entry_adj_open'].where(panel['entry_adj_open'] > 0) - 1
panel['signal_eligible'] = (panel['in_universe'] & panel['paused'].eq(0)
                            & np.isfinite(panel[FEATURES]).all(axis=1))
entry_paused = panel.groupby('stock')['paused'].shift(-1)
exit_paused = panel.groupby('stock')['paused'].shift(-2)
future_paused = panel['signal_eligible'] & (entry_paused.eq(1) | exit_paused.eq(1))
print('可生成信号：', panel['signal_eligible'].sum())
print('有信号但标签不可用：', (panel['signal_eligible'] & ~np.isfinite(panel['label'])).sum())
print('有信号但未来入场或退出日停牌：', future_paused.sum())
display(panel.loc[future_paused, ['date', 'stock', 'entry_date', 'label_end_date']].head())


## 9. 保存建模表

只在这里保存一次数据。02读取这张表，不再读取旧批次或恢复旧变量。
保留全部交易日行，02再按标签成熟日期筛训练样本，按完整窗口筛序列样本。

In [ ]:
columns = ['date', 'stock', 'signal_eligible', 'label', 'label_end_date'] + FEATURES
model_data = panel[columns].copy()
model_data.to_csv(str(OUTPUT / 'model_data.csv.gz'), index=False, compression='gzip', chunksize=10000)
print('建模表：', OUTPUT / 'model_data.csv.gz', '形状：', model_data.shape)
display(model_data.head())
# 保存后可关闭01内核，在02中继续。
